In [3]:
import numpy as np
import pandas as pd
import re
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path


df=Path.cwd().parent.joinpath('projectDM')
crime_df= pd.read_csv('Crime_Data_from_2020_to_Present.csv')
crime_df.head()


,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
0,211507896,4/11/2021 0:00,11/7/2020 0:00,845,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,7800 BEEMAN AV,NaN,34.2124,-118.4092
1,201516622,10/21/2020 0:00,10/18/2020 0:00,1845,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,IC,Invest Cont,230.0,NaN,NaN,NaN,ATOLL AV,N GAULT,34.1993,-118.4203
2,240913563,12/10/2024 0:00,10/30/2020 0:00,1240,9,Van Nuys,933,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,14600 SYLVAN ST,NaN,34.1847,-118.4509
3,210704711,12/24/2020 0:00,12/24/2020 0:00,1310,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,...,IC,Invest Cont,331.0,NaN,NaN,NaN,6000 COMEY AV,NaN,34.0339,-118.3747
4,201418201,10/3/2020 0:00,9/29/2020 0:00,1830,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),...,IC,Invest Cont,420.0,NaN,NaN,NaN,4700 LA VILLA MARINA,NaN,33.9813,-118.4350


In [47]:
crime_df.Status.unique()

['IC', 'AO', 'AA', 'JA', 'JO', 'CC']
Categories (6, object): ['AA', 'AO', 'CC', 'IC', 'JA', 'JO']

In [4]:
crime_df = crime_df.reset_index(drop=True)

In [5]:
crime_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1004991 entries, 0 to 1004990
Data columns (total 28 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   DR_NO           1004991 non-null  int64  
 1   Date Rptd       1004991 non-null  object 
 2   DATE OCC        1004991 non-null  object 
 3   TIME OCC        1004991 non-null  int64  
 4   AREA            1004991 non-null  int64  
 5   AREA NAME       1004991 non-null  object 
 6   Rpt Dist No     1004991 non-null  int64  
 7   Part 1-2        1004991 non-null  int64  
 8   Crm Cd          1004991 non-null  int64  
 9   Crm Cd Desc     1004991 non-null  object 
 10  Mocodes         853372 non-null   object 
 11  Vict Age        1004991 non-null  int64  
 12  Vict Sex        860347 non-null   object 
 13  Vict Descent    860335 non-null   object 
 14  Premis Cd       1004975 non-null  float64
 15  Premis Desc     1004403 non-null  object 
 16  Weapon Used Cd  327247 non-null   fl

In [6]:
pd.set_option('display.max_columns', None)

In [7]:
crime_df["TIME OCC"] = crime_df["TIME OCC"].astype("Int64")

crime_df["hour_occ"] = crime_df["TIME OCC"] // 100
crime_df["min_occ"]  = crime_df["TIME OCC"] % 100

In [8]:
crime_df["TIME OCC"] = crime_df["TIME OCC"].astype(str).str.zfill(4)

In [9]:
crime_df["TIME OCC"] = pd.to_datetime(
    crime_df["TIME OCC"],
    format="%H%M",
    errors="coerce"
).dt.time

In [10]:
import pandas as pd

# 1) Convert dates
crime_df["Date Rptd"] = pd.to_datetime(crime_df["Date Rptd"], errors="coerce")
crime_df["DATE OCC"]  = pd.to_datetime(crime_df["DATE OCC"], errors="coerce")

# 2) Convert ID to string (so it isn't treated as a numeric feature)
crime_df["DR_NO"] = crime_df["DR_NO"].astype(str)

# 3) Convert code columns to category (or string)
code_cols = ["Rpt Dist No", "Crm Cd", "Crm Cd 1", "Premis Cd"]
for c in code_cols:
    crime_df[c] = crime_df[c].astype("Int64")  # keeps missing values nicely
    crime_df[c] = crime_df[c].astype("category")

# 4) Optional: make obvious categoricals category dtype (saves memory)
cat_cols = ["AREA NAME", "Crm Cd Desc", "Vict Sex", "Vict Descent", 
            "Premis Desc", "Status", "Status Desc"]
for c in cat_cols:
    crime_df[c] = crime_df[c].astype("category")

# Check updated types
crime_df.dtypes

DR_NO                     object
Date Rptd         datetime64[ns]
DATE OCC          datetime64[ns]
TIME OCC                  object
AREA                       int64
AREA NAME               category
Rpt Dist No             category
Part 1-2                   int64
Crm Cd                  category
Crm Cd Desc             category
Mocodes                   object
Vict Age                   int64
Vict Sex                category
Vict Descent            category
Premis Cd               category
Premis Desc             category
Weapon Used Cd           float64
Weapon Desc               object
Status                  category
Status Desc             category
Crm Cd 1                category
Crm Cd 2                 float64
Crm Cd 3                 float64
Crm Cd 4                 float64
LOCATION                  object
Cross Street              object
LAT                      float64
LON                      float64
hour_occ                   Int64
min_occ                    Int64
dtype: obj

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

text_data = crime_df["Crm Cd Desc"].astype("string").fillna("")

tfidf = TfidfVectorizer(stop_words="english", max_features=1000)
X_tfidf = tfidf.fit_transform(text_data)

print(X_tfidf.shape)

(1004991, 226)


In [12]:
crime_df.info

<bound method DataFrame.info of              DR_NO  Date Rptd   DATE OCC  TIME OCC  AREA    AREA NAME  \
0        211507896 2021-04-11 2020-11-07  08:45:00    15  N Hollywood   
1        201516622 2020-10-21 2020-10-18  18:45:00    15  N Hollywood   
2        240913563 2024-12-10 2020-10-30  12:40:00     9     Van Nuys   
3        210704711 2020-12-24 2020-12-24  13:10:00     7     Wilshire   
4        201418201 2020-10-03 2020-09-29  18:30:00    14      Pacific   
...            ...        ...        ...       ...   ...          ...   
1004986  252104112 2025-02-02 2025-02-02  01:30:00    21      Topanga   
1004987  250404100 2025-02-18 2025-02-18  10:00:00     4   Hollenbeck   
1004988  251304095 2025-01-31 2025-01-30  15:54:00    13       Newton   
1004989  251704066 2025-01-17 2025-01-17  16:00:00    17   Devonshire   
1004990  251904210 2025-03-25 2025-03-25  12:35:00    19      Mission   

        Rpt Dist No  Part 1-2 Crm Cd  \
0              1502         2    354   
1          

In [13]:
crime_df.duplicated().sum()

np.int64(0)

In [14]:
crime_df.head()

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,Mocodes,Vict Age,Vict Sex,Vict Descent,Premis Cd,Premis Desc,Weapon Used Cd,Weapon Desc,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON,hour_occ,min_occ
0,211507896,2021-04-11,2020-11-07,08:45:00,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,377,31,M,H,501,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354,NaN,NaN,NaN,7800 BEEMAN AV,NaN,34.2124,-118.4092,8,45
1,201516622,2020-10-21,2020-10-18,18:45:00,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",0416 0334 2004 1822 1414 0305 0319 0400,32,M,H,102,SIDEWALK,200.0,KNIFE WITH BLADE 6INCHES OR LESS,IC,Invest Cont,230,NaN,NaN,NaN,ATOLL AV,N GAULT,34.1993,-118.4203,18,45
2,240913563,2024-12-10,2020-10-30,12:40:00,9,Van Nuys,933,2,354,THEFT OF IDENTITY,377,30,M,W,501,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354,NaN,NaN,NaN,14600 SYLVAN ST,NaN,34.1847,-118.4509,12,40
3,210704711,2020-12-24,2020-12-24,13:10:00,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,344,47,F,A,101,STREET,NaN,NaN,IC,Invest Cont,331,NaN,NaN,NaN,6000 COMEY AV,NaN,34.0339,-118.3747,13,10
4,201418201,2020-10-03,2020-09-29,18:30:00,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),1300 0344 1606 2032,63,M,H,103,ALLEY,NaN,NaN,IC,Invest Cont,420,NaN,NaN,NaN,4700 LA VILLA MARINA,NaN,33.9813,-118.4350,18,30


In [15]:
crime_df['Vict Age'].unique()

array([ 31,  32,  30,  47,  63,  35,  21,  14,  43,  57,  13,  34,   0,
        39,  26,  37,  69,  24,  36,  19,  48,  49,  22,  58,  46,  33,
        23,  28,  18,  51,  11,  61,  74,  53,  68,  50,  62,  41,  27,
        17,  60,  52,  29,  40,  59,  25,  80,  44,  42,  65,  70,  66,
        45,  56,  54,  20,   7,  79,  67,   9,  55,  83,  38,  99,  72,
        71,  96,  16,   8,  77,  81,  73,  64,   4,  91,  12,  76,  75,
        15,  82,   3,  89,  10,   6,  86,  90,  78,  85,  84,  87,   2,
        -4,  -3,   5,  88,  95,  -2,  92,  93,  97,  94,  -1,  98, 120])

In [16]:


#crime_df = crime_df[crime_df['Vict Age'] >= 0]

crime_df.isna().sum()




DR_NO                   0
Date Rptd               0
DATE OCC                0
TIME OCC                0
AREA                    0
AREA NAME               0
Rpt Dist No             0
Part 1-2                0
Crm Cd                  0
Crm Cd Desc             0
Mocodes            151619
Vict Age                0
Vict Sex           144644
Vict Descent       144656
Premis Cd              16
Premis Desc           588
Weapon Used Cd     677744
Weapon Desc        677744
Status                  1
Status Desc             0
Crm Cd 1               11
Crm Cd 2           935831
Crm Cd 3          1002677
Crm Cd 4          1004927
LOCATION                0
Cross Street       850755
LAT                     0
LON                     0
hour_occ                0
min_occ                 0
dtype: int64

In [17]:
crime_df = crime_df[(crime_df['Vict Age'] > 0) & (crime_df['Vict Age'] <= 100)]



In [18]:
crime_df['age_group'] = pd.cut(
    crime_df['Vict Age'],
    bins=[0, 13, 18, 35, 55, 100],
    labels=[
        'Child',
        'Adolescent',
        'Young_Adult',
        'Middle_Aged_Adult',
        'Older_Adult'
    ]
)


C:\Users\That black girl\AppData\Local\Temp\ipykernel_13976\4206491345.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  crime_df['age_group'] = pd.cut(


In [45]:
import pandas as pd

age_table = pd.DataFrame({
    "Age Group": [
        "Child",
        "Adolescent",
        "Young Adult",
        "Middle Aged Adult",
        "Older Adult"
    ],
    "Age Range": [
        "0 - 13",
        "14 - 18",
        "19 - 35",
        "36 - 55",
        "56 - 100"
    ]
})

age_table

,Age Group,Age Range
0,Child,0 - 13
1,Adolescent,14 - 18
2,Young Adult,19 - 35
3,Middle Aged Adult,36 - 55
4,Older Adult,56 - 100


In [19]:
age_table = crime_df['age_group'].value_counts().reset_index()
age_table.columns = ['Age Group', 'Number of Victims']

age_table

,Age Group,Number of Victims
0,Young_Adult,315939
1,Middle_Aged_Adult,262632
2,Older_Adult,125388
3,Adolescent,20866
4,Child,10806


In [20]:
crime_df.head()

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,Mocodes,Vict Age,Vict Sex,Vict Descent,Premis Cd,Premis Desc,Weapon Used Cd,Weapon Desc,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON,hour_occ,min_occ,age_group
0,211507896,2021-04-11,2020-11-07,08:45:00,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,377,31,M,H,501,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354,NaN,NaN,NaN,7800 BEEMAN AV,NaN,34.2124,-118.4092,8,45,Young_Adult
1,201516622,2020-10-21,2020-10-18,18:45:00,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",0416 0334 2004 1822 1414 0305 0319 0400,32,M,H,102,SIDEWALK,200.0,KNIFE WITH BLADE 6INCHES OR LESS,IC,Invest Cont,230,NaN,NaN,NaN,ATOLL AV,N GAULT,34.1993,-118.4203,18,45,Young_Adult
2,240913563,2024-12-10,2020-10-30,12:40:00,9,Van Nuys,933,2,354,THEFT OF IDENTITY,377,30,M,W,501,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354,NaN,NaN,NaN,14600 SYLVAN ST,NaN,34.1847,-118.4509,12,40,Young_Adult
3,210704711,2020-12-24,2020-12-24,13:10:00,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,344,47,F,A,101,STREET,NaN,NaN,IC,Invest Cont,331,NaN,NaN,NaN,6000 COMEY AV,NaN,34.0339,-118.3747,13,10,Middle_Aged_Adult
4,201418201,2020-10-03,2020-09-29,18:30:00,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),1300 0344 1606 2032,63,M,H,103,ALLEY,NaN,NaN,IC,Invest Cont,420,NaN,NaN,NaN,4700 LA VILLA MARINA,NaN,33.9813,-118.4350,18,30,Older_Adult


In [21]:
crime_df['Vict Sex'] = crime_df['Vict Sex'].replace({
    'M': 'Male',
    'F': 'Female',
    'X': 'Unknown',
    'H': 'Unknown' 
})

crime_df['Vict Sex'] = crime_df['Vict Sex'].fillna('Unknown')
crime_df['Vict Sex'] = crime_df['Vict Sex'].astype('category')


C:\Users\That black girl\AppData\Local\Temp\ipykernel_13976\1159674280.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  crime_df['Vict Sex'] = crime_df['Vict Sex'].replace({
C:\Users\That black girl\AppData\Local\Temp\ipykernel_13976\1159674280.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  crime_df['Vict Sex'] = crime_df['Vict Sex'].replace({
C:\Users\That black girl\AppData\Local\Temp\ipykernel_13976\1159674280.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row

In [22]:
crime_df['Vict Descent'] = crime_df['Vict Descent'].replace('-', 'Other')
crime_df['Vict Descent'] = crime_df['Vict Descent'].fillna('Other')
crime_df['Vict Descent'] = crime_df['Vict Descent'].astype('category')


C:\Users\That black girl\AppData\Local\Temp\ipykernel_13976\284575008.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  crime_df['Vict Descent'] = crime_df['Vict Descent'].replace('-', 'Other')
C:\Users\That black girl\AppData\Local\Temp\ipykernel_13976\284575008.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  crime_df['Vict Descent'] = crime_df['Vict Descent'].replace('-', 'Other')
C:\Users\That black girl\AppData\Local\Temp\ipykernel_13976\284575008.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice

In [23]:
descent_mapping = {
    'A': 'Asian',
    'B': 'Black',
    'C': 'Chinese',
    'D': 'Cambodian',
    'F': 'Filipino',
    'G': 'Guamanian',
    'H': 'Hispanic',
    'I': 'American Indian',
    'J': 'Japanese',
    'K': 'Korean',
    'L': 'Laotian',
    'O': 'Other',
    'P': 'Pacific Islander',
    'S': 'Samoan',
    'U': 'Hawaiian',
    'V': 'Vietnamese',
    'W': 'White',
    'X': 'Unknown',
    'Z': 'Asian Indian',
    
}

crime_df['Vict Descent'] = crime_df['Vict Descent'].map(descent_mapping)


C:\Users\That black girl\AppData\Local\Temp\ipykernel_13976\3418896396.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  crime_df['Vict Descent'] = crime_df['Vict Descent'].map(descent_mapping)


In [24]:

crime_df.drop(columns=['Crm Cd 4','Crm Cd 3','Crm Cd 2','Cross Street','TIME OCC'],inplace=True)
crime_df.head()


C:\Users\That black girl\AppData\Local\Temp\ipykernel_13976\2455021678.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  crime_df.drop(columns=['Crm Cd 4','Crm Cd 3','Crm Cd 2','Cross Street','TIME OCC'],inplace=True)


,DR_NO,Date Rptd,DATE OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,Mocodes,Vict Age,Vict Sex,Vict Descent,Premis Cd,Premis Desc,Weapon Used Cd,Weapon Desc,Status,Status Desc,Crm Cd 1,LOCATION,LAT,LON,hour_occ,min_occ,age_group
0,211507896,2021-04-11,2020-11-07,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,377,31,Male,Hispanic,501,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354,7800 BEEMAN AV,34.2124,-118.4092,8,45,Young_Adult
1,201516622,2020-10-21,2020-10-18,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",0416 0334 2004 1822 1414 0305 0319 0400,32,Male,Hispanic,102,SIDEWALK,200.0,KNIFE WITH BLADE 6INCHES OR LESS,IC,Invest Cont,230,ATOLL AV,34.1993,-118.4203,18,45,Young_Adult
2,240913563,2024-12-10,2020-10-30,9,Van Nuys,933,2,354,THEFT OF IDENTITY,377,30,Male,White,501,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354,14600 SYLVAN ST,34.1847,-118.4509,12,40,Young_Adult
3,210704711,2020-12-24,2020-12-24,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,344,47,Female,Asian,101,STREET,NaN,NaN,IC,Invest Cont,331,6000 COMEY AV,34.0339,-118.3747,13,10,Middle_Aged_Adult
4,201418201,2020-10-03,2020-09-29,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),1300 0344 1606 2032,63,Male,Hispanic,103,ALLEY,NaN,NaN,IC,Invest Cont,420,4700 LA VILLA MARINA,33.9813,-118.4350,18,30,Older_Adult


In [25]:
type(crime_df)


pandas.core.frame.DataFrame

In [26]:
crime_df['DATE OCC'] = pd.to_datetime(crime_df['DATE OCC'], errors='coerce')
crime_df

C:\Users\That black girl\AppData\Local\Temp\ipykernel_13976\937620469.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  crime_df['DATE OCC'] = pd.to_datetime(crime_df['DATE OCC'], errors='coerce')


,DR_NO,Date Rptd,DATE OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,Mocodes,Vict Age,Vict Sex,Vict Descent,Premis Cd,Premis Desc,Weapon Used Cd,Weapon Desc,Status,Status Desc,Crm Cd 1,LOCATION,LAT,LON,hour_occ,min_occ,age_group
0,211507896,2021-04-11,2020-11-07,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,377,31,Male,Hispanic,501,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354,7800 BEEMAN AV,34.2124,-118.4092,8,45,Young_Adult
1,201516622,2020-10-21,2020-10-18,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",0416 0334 2004 1822 1414 0305 0319 0400,32,Male,Hispanic,102,SIDEWALK,200.0,KNIFE WITH BLADE 6INCHES OR LESS,IC,Invest Cont,230,ATOLL AV,34.1993,-118.4203,18,45,Young_Adult
2,240913563,2024-12-10,2020-10-30,9,Van Nuys,933,2,354,THEFT OF IDENTITY,377,30,Male,White,501,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354,14600 SYLVAN ST,34.1847,-118.4509,12,40,Young_Adult
3,210704711,2020-12-24,2020-12-24,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,344,47,Female,Asian,101,STREET,NaN,NaN,IC,Invest Cont,331,6000 COMEY AV,34.0339,-118.3747,13,10,Middle_Aged_Adult
4,201418201,2020-10-03,2020-09-29,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),1300 0344 1606 2032,63,Male,Hispanic,103,ALLEY,NaN,NaN,IC,Invest Cont,420,4700 LA VILLA MARINA,33.9813,-118.4350,18,30,Older_Adult
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1004986,252104112,2025-02-02,2025-02-02,21,Topanga,2103,2,946,OTHER MISCELLANEOUS CRIME,NaN,35,Male,Unknown,101,STREET,NaN,NaN,IC,Invest Cont,946,22100 ROSCOE BL,34.2259,-118.6126,1,30,Young_Adult
1004987,250404100,2025-02-18,2025-02-18,4,Hollenbeck,479,2,237,CHILD NEGLECT (SEE 300 W.I.C.),1258 0553 0602,11,Male,Black,501,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,237,3500 PERCY ST,34.0277,-118.1979,10,0,Child
1004988,251304095,2025-01-31,2025-01-30,13,Newton,1372,2,850,INDECENT EXPOSURE,NaN,16,Female,Hispanic,101,STREET,NaN,NaN,IC,Invest Cont,850,300 E 53RD ST,33.9942,-118.2701,15,54,Adolescent
1004989,251704066,2025-01-17,2025-01-17,17,Devonshire,1774,2,624,BATTERY - SIMPLE ASSAULT,0400 1259 1822 0356,17,Male,Hispanic,721,HIGH SCHOOL,400.0,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",IC,Invest Cont,624,9600 ZELZAH AV,34.2450,-118.5233,16,0,Adolescent


In [27]:
crime_df['type_of_crime'] = crime_df['Part 1-2']

C:\Users\That black girl\AppData\Local\Temp\ipykernel_13976\682953243.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  crime_df['type_of_crime'] = crime_df['Part 1-2']


In [28]:
crime_df.drop(columns=['Part 1-2'])

,DR_NO,Date Rptd,DATE OCC,AREA,AREA NAME,Rpt Dist No,Crm Cd,Crm Cd Desc,Mocodes,Vict Age,Vict Sex,Vict Descent,Premis Cd,Premis Desc,Weapon Used Cd,Weapon Desc,Status,Status Desc,Crm Cd 1,LOCATION,LAT,LON,hour_occ,min_occ,age_group,type_of_crime
0,211507896,2021-04-11,2020-11-07,15,N Hollywood,1502,354,THEFT OF IDENTITY,377,31,Male,Hispanic,501,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354,7800 BEEMAN AV,34.2124,-118.4092,8,45,Young_Adult,2
1,201516622,2020-10-21,2020-10-18,15,N Hollywood,1521,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",0416 0334 2004 1822 1414 0305 0319 0400,32,Male,Hispanic,102,SIDEWALK,200.0,KNIFE WITH BLADE 6INCHES OR LESS,IC,Invest Cont,230,ATOLL AV,34.1993,-118.4203,18,45,Young_Adult,1
2,240913563,2024-12-10,2020-10-30,9,Van Nuys,933,354,THEFT OF IDENTITY,377,30,Male,White,501,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354,14600 SYLVAN ST,34.1847,-118.4509,12,40,Young_Adult,2
3,210704711,2020-12-24,2020-12-24,7,Wilshire,782,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,344,47,Female,Asian,101,STREET,NaN,NaN,IC,Invest Cont,331,6000 COMEY AV,34.0339,-118.3747,13,10,Middle_Aged_Adult,1
4,201418201,2020-10-03,2020-09-29,14,Pacific,1454,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),1300 0344 1606 2032,63,Male,Hispanic,103,ALLEY,NaN,NaN,IC,Invest Cont,420,4700 LA VILLA MARINA,33.9813,-118.4350,18,30,Older_Adult,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1004986,252104112,2025-02-02,2025-02-02,21,Topanga,2103,946,OTHER MISCELLANEOUS CRIME,NaN,35,Male,Unknown,101,STREET,NaN,NaN,IC,Invest Cont,946,22100 ROSCOE BL,34.2259,-118.6126,1,30,Young_Adult,2
1004987,250404100,2025-02-18,2025-02-18,4,Hollenbeck,479,237,CHILD NEGLECT (SEE 300 W.I.C.),1258 0553 0602,11,Male,Black,501,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,237,3500 PERCY ST,34.0277,-118.1979,10,0,Child,2
1004988,251304095,2025-01-31,2025-01-30,13,Newton,1372,850,INDECENT EXPOSURE,NaN,16,Female,Hispanic,101,STREET,NaN,NaN,IC,Invest Cont,850,300 E 53RD ST,33.9942,-118.2701,15,54,Adolescent,2
1004989,251704066,2025-01-17,2025-01-17,17,Devonshire,1774,624,BATTERY - SIMPLE ASSAULT,0400 1259 1822 0356,17,Male,Hispanic,721,HIGH SCHOOL,400.0,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",IC,Invest Cont,624,9600 ZELZAH AV,34.2450,-118.5233,16,0,Adolescent,2


In [29]:
crime_df = crime_df.reset_index(drop=True)

In [30]:
crime_df.info()




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 735631 entries, 0 to 735630
Data columns (total 27 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   DR_NO           735631 non-null  object        
 1   Date Rptd       735631 non-null  datetime64[ns]
 2   DATE OCC        735631 non-null  datetime64[ns]
 3   AREA            735631 non-null  int64         
 4   AREA NAME       735631 non-null  category      
 5   Rpt Dist No     735631 non-null  category      
 6   Part 1-2        735631 non-null  int64         
 7   Crm Cd          735631 non-null  category      
 8   Crm Cd Desc     735631 non-null  category      
 9   Mocodes         729951 non-null  object        
 10  Vict Age        735631 non-null  int64         
 11  Vict Sex        735631 non-null  category      
 12  Vict Descent    735599 non-null  object        
 13  Premis Cd       735630 non-null  category      
 14  Premis Desc     735402 non-null  cat

In [31]:
crime_df["is_night"] = ((crime_df["hour_occ"] >= 18) | (crime_df["hour_occ"] < 6)).astype(int)


In [32]:
columns_to_drop = ['Mocodes''AREA','Weapon Used Cd','Weapon Desc']

crime_df = crime_df.drop(columns=columns_to_drop, errors='ignore')


In [33]:
crime_df = crime_df.drop(columns=['Part 1-2'], errors='ignore')


In [34]:
crime_df['Vict Sex'] = crime_df['Vict Sex'].fillna('Unknown')
crime_df['Vict Sex'] = crime_df['Vict Sex'].replace({'X': 'Unknown'})


In [35]:
crime_df['AREA NAME'] = crime_df['AREA NAME'].astype('category')
crime_df['age_group'] = crime_df['age_group'].astype('category')
crime_df['Vict Sex'] = crime_df['Vict Sex'].astype('category')


In [36]:
crime_df.isna().sum()


DR_NO               0
Date Rptd           0
DATE OCC            0
AREA                0
AREA NAME           0
Rpt Dist No         0
Crm Cd              0
Crm Cd Desc         0
Mocodes          5680
Vict Age            0
Vict Sex            0
Vict Descent       32
Premis Cd           1
Premis Desc       229
Status              0
Status Desc         0
Crm Cd 1            6
LOCATION            0
LAT                 0
LON                 0
hour_occ            0
min_occ             0
age_group           0
type_of_crime       0
is_night            0
dtype: int64

In [37]:
# Clean AREA NAME
crime_df['AREA NAME'] = crime_df['AREA NAME'].str.title()
crime_df['AREA NAME'] = crime_df['AREA NAME'].astype('category')

# Drop unnecessary location columns
crime_df = crime_df.drop(columns=[
    'AREA',
    'LOCATION',
    'Cross Street'
], errors='ignore')


In [38]:
crime_df.duplicated().head(10)

0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
dtype: bool

In [39]:
crime_df['Crm Cd Desc'] = crime_df['Crm Cd Desc'].str.split('-').str[0].str.strip()

In [40]:
crime_df = crime_df.drop(columns=['Mocodes'])

In [41]:
crime_df.dtypes

DR_NO                    object
Date Rptd        datetime64[ns]
DATE OCC         datetime64[ns]
AREA NAME              category
Rpt Dist No            category
Crm Cd                 category
Crm Cd Desc              object
Vict Age                  int64
Vict Sex               category
Vict Descent             object
Premis Cd              category
Premis Desc            category
Status                 category
Status Desc            category
Crm Cd 1               category
LAT                     float64
LON                     float64
hour_occ                  Int64
min_occ                   Int64
age_group              category
type_of_crime             int64
is_night                  int64
dtype: object

In [42]:
print("Original rows:", crime_df.shape[0])


Original rows: 735631


In [43]:
crime_df.to_csv("Crime_Data_Cleaned_Finalmom.csv", index=False)

In [44]:
crime_df.to_parquet("crime_cleaned.parquet", index=False)